In [1]:
# Data Cleaning - ONS ASHE Salary Data
# cleaning median salary by occupation for 2021-2025

import pandas as pd
import zipfile

# loading all 5 years - same as EDA
all_years = []

filenames = {
    2021: ('ASHE_2021_13Jun2026.zip', 'Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2021.xls'),
    2022: ('ASHE_2022_13Jun2026.zip', 'Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2022.xls'),
    2023: ('ASHE_2023_13Jun2026.zip', 'Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2023.xlsx'),
    2024: ('ASHE_2024_13Jun2026.zip', 'Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2024.xlsx'),
    2025: ('ASHE_2025_13Jun2026.zip', 'ashetable142025provisional/PROV - Occupation SOC20 (4) Table 14.7a   Annual pay - Gross 2025.xlsx'),
}

for year, (zipname, fname) in filenames.items():
    with zipfile.ZipFile(zipname, 'r') as z:
        with z.open(fname) as f:
            df_year = pd.read_excel(f, sheet_name='All', skiprows=4)

    df_year = df_year[['Description', 'Code', 'Median']].copy()
    df_year = df_year.dropna(subset=['Code'])
    df_year['Median'] = pd.to_numeric(df_year['Median'], errors='coerce')
    df_year = df_year.dropna(subset=['Median'])
    df_year['Year'] = year
    all_years.append(df_year)
    print(f"{year} loaded - {len(df_year)} occupations")

df_ashe = pd.concat(all_years, ignore_index=True)
print("\ntotal rows:", len(df_ashe))

2021 loaded - 490 occupations
2022 loaded - 458 occupations
2023 loaded - 483 occupations
2024 loaded - 496 occupations
2025 loaded - 498 occupations

total rows: 2425


In [2]:
# mapping occupations to my 5 sectors using keywords
sector_keywords = {
    'Technology': 'Information Technology Professionals',
    'Healthcare': 'Health professionals',
    'Finance': 'Finance Professionals',
    'Engineering': 'Engineering Professionals',
    'Education': 'Teaching Professionals'
}

# adding sector column based on description keyword match
def map_sector(description):
    for sector, keyword in sector_keywords.items():
        if keyword.lower() in str(description).lower():
            return sector
    return None

df_ashe['Sector'] = df_ashe['Description'].apply(map_sector)

# keeping only my 5 sectors
df_sectors = df_ashe[df_ashe['Sector'].notna()].copy()

print("shape after sector filtering:", df_sectors.shape)
print("\nsector counts:")
print(df_sectors['Sector'].value_counts())

shape after sector filtering: (80, 5)

sector counts:
Sector
Education      36
Healthcare     19
Engineering    10
Technology     10
Finance         5
Name: count, dtype: int64


In [3]:
# aggregating to median salary per sector per year
df_clean = df_sectors.groupby(['Year', 'Sector'])['Median'].mean().reset_index()
df_clean.columns = ['Year', 'Sector', 'Median_Salary']
df_clean['Median_Salary'] = df_clean['Median_Salary'].round(0).astype(int)

print("median salary per sector per year:")
print(df_clean.pivot(index='Year', columns='Sector', values='Median_Salary'))

median salary per sector per year:
Sector  Education  Engineering  Finance  Healthcare  Technology
Year                                                           
2021        35969        40984    37067       34887       40826
2022        32884        42482    38818       35485       43005
2023        34274        44571    40604       36898       46042
2024        38521        46972    44345       39732       51024
2025        40837        49155    47173       40318       52908


In [4]:
# saving clean file
df_clean.to_csv('ASHE_Clean.csv', index=False)
print("saved! shape:", df_clean.shape)
print("\nfirst 10 rows:")
print(df_clean.head(10))

saved! shape: (25, 3)

first 10 rows:
   Year       Sector  Median_Salary
0  2021    Education          35969
1  2021  Engineering          40984
2  2021      Finance          37067
3  2021   Healthcare          34887
4  2021   Technology          40826
5  2022    Education          32884
6  2022  Engineering          42482
7  2022      Finance          38818
8  2022   Healthcare          35485
9  2022   Technology          43005
